# Figure Generation
## Analysis: Bone Health Outcomes
All figures are generated from pre-computed results in
`results/` and the clean dataset in
`data/processed/BD_Clean_Osteoporosis.parquet`.
No statistical models are re-estimated here.


In [1]:
# Figure 1 — ROC, calibration, forest plot

import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import numpy as np
import pandas as pd
from matplotlib.gridspec import GridSpec

# Confirm relative paths for notebook execution from project root or notebooks/
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTS_DIR = PROJECT_DIR / "results"
OUTPUTS_DIR = PROJECT_DIR / "outputs"

os.makedirs(OUTPUTS_DIR, exist_ok=True)

sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / "src"))

from src.export_tables_pdf import build_model_outputs, load_clean_data

# ── Load data ────────────────────────────────────────────────────────────────
df_clean = load_clean_data()
outputs = build_model_outputs(df_clean)

fpr = outputs["fpr"]
tpr = outputs["tpr"]

roc_df = pd.DataFrame({"fpr": fpr, "tpr": tpr})
roc_df.to_csv(RESULTS_DIR / "roc_curve_data.csv", index=False)

auc_table = pd.read_csv(RESULTS_DIR / "tabla_10_auc_bootstrap.csv", index_col=0)
auc_app = auc_table.loc["AUC aparente", "Valor"]
auc_ci = auc_table.loc["IC 95% bootstrap del AUC aparente", "Valor"]
auc_corrected = auc_table.loc["AUC corregida por optimismo", "Valor"]

calibration_df = pd.read_csv(RESULTS_DIR / "tabla_8_calibracion_deciles.csv")
cal = pd.read_csv(RESULTS_DIR / "tabla_9_calibracion_resumen.csv", index_col=0)
brier = cal.loc["Brier score", "Valor"]
interc = cal.loc["Intercepto de calibración", "Valor"]
slope = cal.loc["Pendiente de calibración", "Valor"]

forest_df = pd.read_csv(RESULTS_DIR / "tabla_5_modelo_logistico_principal.csv")
forest_df = forest_df.loc[forest_df["Variable"] != "Intercepto"].copy()

label_map = {
    "Edad": "Age (years)",
    "IMC": "BMI (kg/m²)",
    "Sexo: mujer vs hombre": "Sex: female vs. male",
    "Trabaja: sí vs no": "Employment: yes vs. no",
    "Enfermedad: sí vs sano": "Disease: yes vs. no",
    "Actividad física: sí vs no": "Physical activity: yes vs. no",
}
forest_df["Label"] = forest_df["Variable"].map(label_map)

if forest_df["Label"].isna().any():
    missing_labels = forest_df.loc[forest_df["Label"].isna(), "Variable"].tolist()
    raise ValueError(f"Missing English labels for variables: {missing_labels}")

forest_df["OR_num"] = pd.to_numeric(forest_df["OR"])
forest_df["CI_low_num"] = pd.to_numeric(forest_df["IC 95% Inferior"])
forest_df["CI_high_num"] = pd.to_numeric(forest_df["IC 95% Superior"])

def is_significant(p_str):
    if str(p_str).strip().startswith("<"):
        return True
    try:
        return float(p_str) < 0.05
    except Exception:
        return False

PDF_DPI = 300
PNG_DPI = 600

required_files = [
    PROJECT_DIR / "data/processed/BD_Clean_Osteoporosis.parquet",
    RESULTS_DIR / "tabla_10_auc_bootstrap.csv",
    RESULTS_DIR / "tabla_8_calibracion_deciles.csv",
    RESULTS_DIR / "tabla_9_calibracion_resumen.csv",
    RESULTS_DIR / "tabla_5_modelo_logistico_principal.csv",
    RESULTS_DIR / "tabla_3_bivariado_continuas.csv",
]
missing = [f for f in required_files if not f.exists()]
if missing:
    raise FileNotFoundError(
        "Required files not found:\n" +
        "\n".join(str(f) for f in missing)
    )
else:
    print("All required files found. Ready to generate figures.")

print(f"Project directory: {PROJECT_DIR}")
print(f"Results available: {list(RESULTS_DIR.glob('*.csv'))}")


All required files found. Ready to generate figures.
Project directory: c:\Users\marco\Documents\analysis_osteoporosis
Results available: [WindowsPath('c:/Users/marco/Documents/analysis_osteoporosis/results/roc_curve_data.csv'), WindowsPath('c:/Users/marco/Documents/analysis_osteoporosis/results/tabla_10_auc_bootstrap.csv'), WindowsPath('c:/Users/marco/Documents/analysis_osteoporosis/results/tabla_1_normalidad.csv'), WindowsPath('c:/Users/marco/Documents/analysis_osteoporosis/results/tabla_2_descriptivo_continuas.csv'), WindowsPath('c:/Users/marco/Documents/analysis_osteoporosis/results/tabla_3_bivariado_continuas.csv'), WindowsPath('c:/Users/marco/Documents/analysis_osteoporosis/results/tabla_4_bivariado_categoricas.csv'), WindowsPath('c:/Users/marco/Documents/analysis_osteoporosis/results/tabla_5_modelo_logistico_principal.csv'), WindowsPath('c:/Users/marco/Documents/analysis_osteoporosis/results/tabla_6_vif.csv'), WindowsPath('c:/Users/marco/Documents/analysis_osteoporosis/results/t

In [2]:
# Participant flow diagram — CONSORT style
# CONSORT flowchart — added

import os
import sys
from pathlib import Path

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_DIR = PROJECT_DIR / "outputs"
MPLCONFIG_DIR = OUTPUT_DIR / ".matplotlib"
OUTPUT_DIR.mkdir(exist_ok=True)
MPLCONFIG_DIR.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIG_DIR.resolve()))

import matplotlib

IN_NOTEBOOK = "ipykernel" in sys.modules
if not IN_NOTEBOOK:
    matplotlib.use("Agg", force=True)

import matplotlib.pyplot as plt
import matplotlib.patches as patches

FONT_FAMILY = "DejaVu Serif"
MAIN_X = 3.25
EXCLUSION_X = 7.7
MAIN_WIDTH = 3.65
MAIN_HEIGHT = 0.86
EXCLUSION_WIDTH = 3.85
EXCLUSION_HEIGHT = 0.86
BOX_LINEWIDTH = 1.05
FINAL_LINEWIDTH = 2.0
ARROW_LINEWIDTH = 1.0

main_steps = [
    ("Initial program participants\n(n = 621)", 10.8),
    ("After exclusion for\nanthropometric data\n(n = 506)", 9.25),
    ("After exclusion\nfor age\n(n = 495)", 7.7),
    ("After exclusion for\nsocioeconomic data\n(n = 420)", 6.15),
    ("After exclusion for\nbone densitometry results\n(n = 408)", 4.6),
    ("Final analytical sample\n(n = 405)", 3.05),
]

exclusions = [
    ("115 excluded\n(Incomplete anthropometric data:\nBMI, weight, height)", 9.98),
    ("11 excluded\n(Incomplete age data)", 8.43),
    ("75 excluded\n(Missing socioeconomic\ninformation)", 6.88),
    ("12 excluded\n(Missing bone densitometry\nresults)", 5.33),
    ("3 excluded\n(Incomplete nutritional\nassessment)", 3.78),
]

fig, ax = plt.subplots(figsize=(8.8, 11.0))
ax.set_xlim(0.7, 10.1)
ax.set_ylim(2.2, 11.7)
ax.axis("off")


def create_simple_box(
    ax,
    center,
    text,
    width,
    height,
    linewidth=BOX_LINEWIDTH,
    fontsize=10.9,
):
    x, y = center
    box = patches.Rectangle(
        (x - width / 2, y - height / 2),
        width,
        height,
        linewidth=linewidth,
        edgecolor="black",
        facecolor="white",
    )
    ax.add_patch(box)
    ax.text(
        x,
        y,
        text,
        ha="center",
        va="center",
        fontsize=fontsize,
        fontfamily=FONT_FAMILY,
        linespacing=1.15,
    )
    return box


def draw_arrow(start, end):
    ax.annotate(
        "",
        xy=end,
        xytext=start,
        arrowprops=dict(
            arrowstyle="->",
            lw=ARROW_LINEWIDTH,
            color="black",
            shrinkA=0,
            shrinkB=0,
            mutation_scale=10,
        ),
    )


for label, y in main_steps[:-1]:
    create_simple_box(ax, (MAIN_X, y), label, MAIN_WIDTH, MAIN_HEIGHT)

final_label, final_y = main_steps[-1]
create_simple_box(
    ax,
    (MAIN_X, final_y),
    final_label,
    MAIN_WIDTH,
    MAIN_HEIGHT,
    linewidth=FINAL_LINEWIDTH,
)

for label, y in exclusions:
    create_simple_box(
        ax,
        (EXCLUSION_X, y),
        label,
        EXCLUSION_WIDTH,
        EXCLUSION_HEIGHT,
        fontsize=10.4,
    )

for (_, y_top), (_, y_bottom) in zip(main_steps, main_steps[1:]):
    draw_arrow((MAIN_X, y_top - MAIN_HEIGHT / 2), (MAIN_X, y_bottom + MAIN_HEIGHT / 2))

for (_, y), (_, y_next) in zip(main_steps, main_steps[1:]):
    midpoint_y = (y + y_next) / 2
    draw_arrow(
        (MAIN_X, midpoint_y),
        (EXCLUSION_X - EXCLUSION_WIDTH / 2, midpoint_y),
    )

plt.savefig(OUTPUT_DIR / "figure_s1_consort_flowchart.pdf",
            bbox_inches="tight", dpi=300)
plt.savefig(OUTPUT_DIR / "figure_s1_consort_flowchart.png",
            bbox_inches="tight", dpi=600)

if IN_NOTEBOOK:
    plt.show()
else:
    plt.close(fig)

C:\Users\marco\AppData\Local\Temp\ipykernel_8024\1167457072.py:145: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [3]:
# ── Figure 1 — ROC curve and calibration plot ─────────────────────────────
fig1, (ax_roc, ax_cal) = plt.subplots(
    1, 2,
    figsize=(12, 5),
    gridspec_kw={"wspace": 0.38}
)

# Panel A — ROC curve
youden_idx = np.argmax(tpr - fpr)

ax_roc.plot(fpr, tpr, color="#2C5F8A", linewidth=1.8)
ax_roc.fill_between(fpr, tpr, color="#2C5F8A", alpha=0.10)
ax_roc.plot([0, 1], [0, 1], color="gray", linestyle=":", linewidth=1)
ax_roc.scatter(
    fpr[youden_idx],
    tpr[youden_idx],
    s=60,
    color="#2C5F8A",
    zorder=3,
)
ax_roc.text(
    0.05,
    0.20,
    f"AUC = {auc_app} (95% CI: {str(auc_ci).replace(' - ', '–')})\n"
    f"Optimism-corrected AUC = {auc_corrected}",
    transform=ax_roc.transAxes,
    fontsize=8,
    va="bottom",
)
ax_roc.set_xlabel("1 − Specificity")
ax_roc.set_ylabel("Sensitivity")
ax_roc.set_title("A", loc="left", fontweight="bold")
ax_roc.set_xlim(0, 1)
ax_roc.set_ylim(0, 1)
ax_roc.set_aspect("equal")

# Panel B — Calibration plot
x_cal = pd.to_numeric(calibration_df["Probabilidad predicha media"])
y_cal = pd.to_numeric(calibration_df["Proporción observada"])
n_cal = pd.to_numeric(calibration_df["n"])

# Ajuste 2 — jitter deciles 6-10
x_vals = x_cal.to_numpy(copy=True)
y_vals = y_cal.to_numpy(copy=True)
x_jittered = x_vals.copy()
for i in range(1, len(x_vals)):
    if abs(x_vals[i] - x_vals[i - 1]) < 0.03:
        direction = 1 if i % 2 == 0 else -1
        x_jittered[i] = x_vals[i] + direction * 0.012

ax_cal.plot(
    [0, 1],
    [0, 1],
    color="gray",
    linestyle=":",
    linewidth=1,
    label="Perfect calibration",
)
ax_cal.scatter(
    x_jittered,
    y_cal,
    s=n_cal * 1.5,
    color="#2C5F8A",
    edgecolors="#2C5F8A",
    linewidth=0.8,
    zorder=2,
)
ax_cal.scatter(
    x_vals[[1, 2]],
    y_vals[[1, 2]],
    s=n_cal.iloc[[1, 2]] * 1.5,
    facecolors="none",
    edgecolors="#C0392B",
    linewidth=1.5,
    zorder=3,
)

# Ajuste 3 — labels deciles 2 and 3
ax_cal.annotate(
    "2",
    xy=(x_vals[1], y_vals[1]),
    xytext=(-10, 6),
    textcoords="offset points",
    fontsize=8,
    color="#C0392B",
    fontweight="normal",
)
ax_cal.annotate(
    "3",
    xy=(x_vals[2], y_vals[2]),
    xytext=(4, -12),
    textcoords="offset points",
    fontsize=8,
    color="#C0392B",
    fontweight="normal",
)

ax_cal.plot(x_jittered, y_cal, color="#2C5F8A", linewidth=1, alpha=0.5)
ax_cal.text(
    0.05,
    0.20,
    # Ajuste 1 — intercept sign
    f"Brier score = {brier}\nIntercept = {abs(float(interc)):.3f}  Slope = {float(slope):.3f}",
    transform=ax_cal.transAxes,
    fontsize=8,
    va="bottom",
)
ax_cal.set_xlabel("Mean predicted probability")
ax_cal.set_ylabel("Observed proportion")
ax_cal.set_title("B", loc="left", fontweight="bold")
ax_cal.set_xlim(0, 1)
ax_cal.set_ylim(0, 1)

for ax in [ax_roc, ax_cal]:
    ax.tick_params(labelsize=9)
    ax.xaxis.label.set_size(10)
    ax.yaxis.label.set_size(10)
    for spine in ["right", "top"]:
        ax.spines[spine].set_visible(False)

fig1.suptitle(
    "Figure 1. Model performance: discrimination and calibration",
    fontsize=11,
    fontweight="normal",
    fontstyle="italic",
    y=1.02,
)

fig1.savefig(
    OUTPUTS_DIR / "figure1_roc_calibration.pdf",
    bbox_inches="tight",
    dpi=PDF_DPI,
    facecolor="white",
)
fig1.savefig(
    OUTPUTS_DIR / "figure1_roc_calibration.png",
    bbox_inches="tight",
    dpi=PNG_DPI,
    facecolor="white",
)
plt.show()


C:\Users\marco\AppData\Local\Temp\ipykernel_8024\1104281132.py:141: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
# ── Figure 2 — Forest plot of odds ratios ─────────────────────────────────
fig2, ax_fp = plt.subplots(figsize=(10, 5))
fig2.subplots_adjust(right=0.72)

# Panel A — Forest plot
y_pos = np.arange(len(forest_df))
colors = [
    "#2C5F8A" if is_significant(p_value) else "#7F8C8D"
    for p_value in forest_df["Valor p"]
]

ax_fp.axvline(1.0, color="gray", linestyle=":", linewidth=1)

for idx, (_, row) in enumerate(forest_df.iterrows()):
    color = colors[idx]
    ax_fp.errorbar(
        row["OR_num"],
        y_pos[idx],
        xerr=[
            [row["OR_num"] - row["CI_low_num"]],
            [row["CI_high_num"] - row["OR_num"]],
        ],
        fmt="s",
        markersize=np.sqrt(50),
        linewidth=1.5,
        color=color,
        ecolor=color,
        capsize=0,
    )

x_text = forest_df["CI_high_num"].max() * 1.15
for idx, (_, row) in enumerate(forest_df.iterrows()):
    or_val = row["OR_num"]
    ic_inf = row["CI_low_num"]
    ic_sup = row["CI_high_num"]
    ax_fp.text(
        x_text,
        y_pos[idx],
        f"OR {float(or_val):.2f} ({float(ic_inf):.2f}–{float(ic_sup):.2f})",
        va="center",
        fontsize=8,
        color=colors[idx],
    )

x_min = forest_df["CI_low_num"].min() * 0.80
x_max = x_text * 2.10

ax_fp.set_xscale("log")
ax_fp.set_xlim(x_min, x_max)
ax_fp.set_yticks(y_pos)
ax_fp.set_yticklabels(forest_df["Label"])
ax_fp.set_xlabel("Odds Ratio (95% CI)")
ax_fp.invert_yaxis()

ax_fp.tick_params(labelsize=9)
ax_fp.xaxis.label.set_size(10)
ax_fp.yaxis.label.set_size(10)
for spine in ["right", "top"]:
    ax_fp.spines[spine].set_visible(False)

fig2.suptitle(
    "Figure 2. Independent associations with bone alteration: "
    "multivariable logistic regression",
    fontsize=11,
    fontweight="normal",
    fontstyle="italic",
    y=1.02,
)

fig2.savefig(
    OUTPUTS_DIR / "figure2_forest_plot.pdf",
    bbox_inches="tight",
    dpi=PDF_DPI,
    facecolor="white",
)
fig2.savefig(
    OUTPUTS_DIR / "figure2_forest_plot.png",
    bbox_inches="tight",
    dpi=PNG_DPI,
    facecolor="white",
)
plt.show()

C:\Users\marco\AppData\Local\Temp\ipykernel_8024\999728932.py:82: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
# Figure 3 — BMI and age distribution by bone health — added
# Data source: clean dataset — no hardcoded values

import os
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTS_DIR = PROJECT_DIR / "results"
OUTPUTS_DIR = PROJECT_DIR / "outputs"

os.makedirs(OUTPUTS_DIR, exist_ok=True)

# ── Load data ────────────────────────────────────────────────────────────────
df = pd.read_parquet(PROJECT_DIR / "data/processed/BD_Clean_Osteoporosis.parquet")

col_imc = "imc"
col_edad = "edad"
col_grupo = "alteracion_osea"  # 0 = normal, 1 = alteración

group_normal_bmi = df[df[col_grupo] == 0][col_imc].dropna()
group_altered_bmi = df[df[col_grupo] == 1][col_imc].dropna()
group_normal_age = df[df[col_grupo] == 0][col_edad].dropna()
group_altered_age = df[df[col_grupo] == 1][col_edad].dropna()

n_normal = int((df[col_grupo] == 0).sum())
n_altered = int((df[col_grupo] == 1).sum())

# columns: ['Variable', 'Alteración ósea\nMediana (Q1, Q3)', 'Salud ósea normal\nMediana (Q1, Q3)', 'U', 'p-value', 'Delta de Cliff', 'Magnitud']
bivariado = pd.read_csv(RESULTS_DIR / "tabla_3_bivariado_continuas.csv")

cliff_imc = bivariado.loc[
    bivariado["Variable"] == "IMC", "Delta de Cliff"
].values[0]
cliff_edad = bivariado.loc[
    bivariado["Variable"] == "Edad (años)", "Delta de Cliff"
].values[0]

mag_imc = bivariado.loc[
    bivariado["Variable"] == "IMC", "Magnitud"
].values[0]
mag_edad = bivariado.loc[
    bivariado["Variable"] == "Edad (años)", "Magnitud"
].values[0]

magnitude_map = {
    "Grande": "Large",
    "Mediano": "Moderate",
    "Pequeño": "Small",
    "Trivial": "Trivial",
}

# ── Helpers ──────────────────────────────────────────────────────────────────
def draw_violin_box_jitter(ax, group_normal, group_altered, ylabel, panel_label):
    groups = [group_normal.values, group_altered.values]

    parts = ax.violinplot(
        groups,
        positions=[0, 1],
        showmedians=False,
        showextrema=False,
        widths=0.6,
    )
    for pc in parts["bodies"]:
        pc.set_facecolor("#AEC6D8")
        pc.set_edgecolor("#2C5F8A")
        pc.set_alpha(0.5)
        pc.set_linewidth(0.8)

    ax.boxplot(
        groups,
        positions=[0, 1],
        widths=0.12,
        patch_artist=True,
        showfliers=False,
        medianprops=dict(color="#2C5F8A", linewidth=2),
        boxprops=dict(facecolor="white", edgecolor="#2C5F8A", linewidth=1),
        whiskerprops=dict(color="#2C5F8A", linewidth=1),
        capprops=dict(color="#2C5F8A", linewidth=1),
    )

    for i, group_data in enumerate([group_normal, group_altered]):
        x_jitter = np.random.normal(i, 0.06, size=len(group_data))
        ax.scatter(
            x_jitter,
            group_data.values,
            alpha=0.25,
            s=8,
            color="#2C5F8A",
            zorder=2,
        )

    ax.set_xticks([0, 1])
    ax.set_xticklabels(
        [
            f"Normal bone health\n(n = {n_normal})",
            f"Bone alteration\n(n = {n_altered})",
        ],
        fontsize=9,
    )
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_title(panel_label, loc="left", fontweight="bold", fontsize=12)


# ── Figure layout ────────────────────────────────────────────────────────────
fig3, (ax_bmi, ax_age) = plt.subplots(
    1,
    2,
    figsize=(11, 6),
    gridspec_kw={"wspace": 0.38},
)

np.random.seed(42)

# Panel A — BMI
draw_violin_box_jitter(
    ax_bmi,
    group_normal_bmi,
    group_altered_bmi,
    "BMI (kg/m²)",
    "A",
)

y_max_bmi = max(group_normal_bmi.max(), group_altered_bmi.max())
y_ann_bmi = y_max_bmi * 1.05
ax_bmi.plot([0, 1], [y_ann_bmi, y_ann_bmi], color="black", lw=0.8)
ax_bmi.text(
    0.5,
    y_ann_bmi * 1.01,
    f"δ = {float(cliff_imc):.3f} ({magnitude_map.get(mag_imc, mag_imc)}) · p < 0.001",
    ha="center",
    va="bottom",
    fontsize=8,
)
ax_bmi.set_ylim(group_normal_bmi.min() * 0.92, y_ann_bmi * 1.08)

# Panel B — Age
draw_violin_box_jitter(
    ax_age,
    group_normal_age,
    group_altered_age,
    "Age (years)",
    "B",
)

y_max_age = max(group_normal_age.max(), group_altered_age.max())
y_ann_age = y_max_age * 1.05
ax_age.plot([0, 1], [y_ann_age, y_ann_age], color="black", lw=0.8)
ax_age.text(
    0.5,
    y_ann_age * 1.01,
    f"δ = {float(cliff_edad):.3f} ({magnitude_map.get(mag_edad, mag_edad)}) · p < 0.001",
    ha="center",
    va="bottom",
    fontsize=8,
)
ax_age.set_ylim(group_normal_age.min() * 0.92, y_ann_age * 1.08)

# ── Global style and export ──────────────────────────────────────────────────
for ax in [ax_bmi, ax_age]:
    ax.tick_params(labelsize=9)
    for spine in ["right", "top"]:
        ax.spines[spine].set_visible(False)
    ax.set_xlim(-0.5, 1.5)

fig3.suptitle(
    "Figure 3. Distribution of BMI and age by bone health status",
    fontsize=11,
    fontweight="normal",
    fontstyle="italic",
    y=1.02,
)

fig3.savefig(
    OUTPUTS_DIR / "figure3_bmi_age_distribution.pdf",
    bbox_inches="tight",
    dpi=300,
    facecolor="white",
)
fig3.savefig(
    OUTPUTS_DIR / "figure3_bmi_age_distribution.png",
    bbox_inches="tight",
    dpi=600,
    facecolor="white",
)
plt.show()

C:\Users\marco\AppData\Local\Temp\ipykernel_8024\2128557466.py:190: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
# Figure 4 — Prevalence by age group and sex — added

import os
from pathlib import Path

try:
    PROJECT_DIR = Path(__file__).resolve().parents[1]
except NameError:
    PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_DIR / ".cache" / "matplotlib"))

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import FixedLocator, FuncFormatter


OUTPUTS_DIR = PROJECT_DIR / "outputs"
DATA_PATH = PROJECT_DIR / "data" / "processed" / "BD_Clean_Osteoporosis.parquet"


def main():
    OUTPUTS_DIR.mkdir(exist_ok=True)

    df = pd.read_parquet(DATA_PATH)
    bins = [49, 59, 69, 200]
    # Cambio 1 — traducción
    labels = ["50–59 years", "60–69 years", "≥70 years"]
    age_groups = labels
    df["age_group"] = pd.cut(df["edad"], bins=bins, labels=labels)
    sex_map = {0: "Male", 1: "Female"}
    df["sex_label"] = df["sexo"].map(sex_map)

    summary = (
        df.groupby(["age_group", "sex_label"], observed=True)
        .agg(
            n_total=("alteracion_osea", "count"),
            n_alteration=("alteracion_osea", "sum"),
        )
        .reset_index()
    )
    summary["pct_alteration"] = (
        summary["n_alteration"] / summary["n_total"] * 100
    )
    total_prev = df["alteracion_osea"].mean() * 100

    plt.rcParams.update(
        {
            "figure.facecolor": "white",
            "axes.facecolor": "white",
            "font.family": "DejaVu Sans",
            "axes.labelcolor": "#222222",
            "xtick.color": "#222222",
            "ytick.color": "#222222",
            "text.color": "#222222",
        }
    )

    colors = {
        "Male": "#78A9CF",
        "Female": "#0756B5",
    }
    sexes = ["Male", "Female"]
    x = np.arange(len(age_groups))
    # Cambio 5 — ancho barras
    bar_width = 0.35

    fig4, ax = plt.subplots(figsize=(8.4, 5.4), dpi=300)
    fig4.subplots_adjust(left=0.105, right=0.88, top=0.84, bottom=0.17)

    for i, sex in enumerate(sexes):
        subset = summary[summary["sex_label"] == sex].set_index("age_group")
        pct = [subset.loc[age_group, "pct_alteration"] for age_group in age_groups]
        n_vals = [subset.loc[age_group, "n_total"] for age_group in age_groups]
        offset = (i - 0.5) * bar_width

        bars = ax.bar(
            x + offset,
            pct,
            width=bar_width,
            color=colors[sex],
            edgecolor="white",
            linewidth=0.8,
            label=sex,
            zorder=3,
        )

        # Cambio 4 — anotaciones
        for bar, pct_val, n_val in zip(bars, pct, n_vals):
            caution = "†" if n_val < 20 else ""
            # Posición Y dinámica
            y_pos_text = bar.get_height() + 0.8
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                y_pos_text,
                f"{pct_val:.1f}%{caution}",
                ha="center",
                va="bottom",
                fontsize=8,
                fontweight="normal",
                color=colors[sex]
            )
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                y_pos_text + 3.5,
                f"(n = {n_val})",
                ha="center",
                va="bottom",
                fontsize=7,
                color="gray"
            )

    # Cambio 3 — línea referencia
    ax.axhline(
        total_prev,
        color="gray",
        linestyle="--",
        linewidth=0.8,
        alpha=0.6,
        zorder=1
    )
    ax.text(
        1.01,
        total_prev,
        f"Overall: {total_prev:.1f}%",
        transform=ax.get_yaxis_transform(),
        fontsize=7.5,
        color="gray",
        ha="left",
        va="center",
        bbox={"facecolor": "white", "edgecolor": "none", "pad": 1.5},
        clip_on=False
    )

    ax.set_xlim(-0.55, len(age_groups) - 0.45)
    ax.set_ylim(0, 100)
    ax.set_xticks(x)
    ax.set_xticklabels(age_groups, fontsize=10, fontfamily="DejaVu Sans")
    # Cambio 1 — traducción
    ax.set_xlabel("Age group", fontsize=10)
    ax.set_ylabel("Prevalence of bone alteration (%)", fontsize=10)
    ax.yaxis.set_major_locator(FixedLocator([0, 20, 40, 60, 80, 100]))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda val, _: f"{val:.0f}%"))

    ax.legend(
        loc="upper left",
        bbox_to_anchor=(0.012, 0.988),
        borderaxespad=0,
        frameon=False,
        fontsize=9,
        handlelength=1.2,
        handleheight=0.8,
        labelspacing=0.45,
    )

    ax.tick_params(axis="both", which="major", length=0, pad=6)
    ax.set_axisbelow(True)
    ax.yaxis.grid(True, color="#D7DCE2", linestyle="-", linewidth=0.7)
    ax.xaxis.grid(False)

    for spine in ["top", "right", "left"]:
        ax.spines[spine].set_visible(False)
    ax.spines["bottom"].set_color("#222222")
    ax.spines["bottom"].set_linewidth(0.8)

    # Cambio 2 — sin título incrustado
    fig4.suptitle(
        "",
        fontsize=1
    )

    # Cambio 6 — nota cautela
    ax.text(
        0.01, -0.12,
        "† Interpret with caution: n < 20 (Male, 50–59 years)",
        transform=ax.transAxes,
        fontsize=7.5,
        color="gray",
        va="bottom"
    )

    for ext, dpi in [("pdf", 300), ("png", 600)]:
        fig4.savefig(
            OUTPUTS_DIR / f"figure4_prevalence_age_sex.{ext}",
            bbox_inches="tight",
            dpi=dpi,
            facecolor="white",
        )
    plt.close(fig4)


if __name__ == "__main__":
    main()
